# Week 7 Assignment - Document Question Answering System (RAG)


**Retrieval-Augmented Generation** Pipeline


| Stage | Tool |
|---|---|
| PDF ingestion | PyMuPDF |
| Text chunking | Custom sliding-window splitter |
| Embeddings | Cohere embed-english-v3.0 |
| Vector store | Pinecone (serverless) |
| Answer generation | command-a-plus-05-2026 |

## 0. Install Dependencies

In [119]:
#!pip install pinecone cohere PyMuPDF --quiet

## 1. Imports & API Keys

In [ ]:
import os, re, time, textwrap, uuid
import fitz                          # PyMuPDF
import cohere
from pinecone import Pinecone, ServerlessSpec

# API Keys
COHERE_API_KEY  = "REDACTED"    # ← paste your key here
PINECONE_API_KEY = "REDACTED" # ← paste your key here


co = cohere.ClientV2(COHERE_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

# Hyper-parameters
CHUNK_SIZE    = 300          # words per chunk
CHUNK_OVERLAP = 50           # word overlap between chunks
TOP_K         = 5            # chunks to retrieve per query
EMBED_MODEL   = "embed-english-v3.0"   # Cohere embedding model
GEN_MODEL     = "command-a-plus-05-2026" # Cohere generation model
INDEX_NAME    = "rag-week7"            # Pinecone index name
EMBED_DIM     = 1024                   # embed-english-v3.0 output dim

print("Clients initialised.")

Clients initialised.


## 2. Document Ingestion
Supports **PDF** and plain **text / markdown** files.


In [ ]:
def load_document(path: str) -> str:
    """Load a PDF or text file and return cleaned raw text."""
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".pdf":
        doc = fitz.open(path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
    elif ext in (".txt", ".md"):
        with open(path, encoding="utf-8") as f:
            text = f.read()
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    print(f"Loaded '{path}' — {len(text):,} characters")
    return text

## 3. Text Chunking

Sliding-window split with overlap so no fact gets cut at a boundary.


In [122]:
def chunk_text(text: str,
               chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Split `text` into overlapping word-windows.

    Parameters
    ----------
    text       : raw document string
    chunk_size : target words per chunk
    overlap    : words shared between consecutive chunks

    Returns
    -------
    List of text chunk strings
    """
    words = text.split()
    step  = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    print(f"Created {len(chunks)} chunks "
          f"(size={chunk_size} words, overlap={overlap} words)")
    return chunks

## 4. Embedding with Cohere
embed-english-v3.0 produces 1024-dimensional vectors.


In [123]:
def embed_documents(chunks: list[str]) -> list[list[float]]:
    """Embed a list of document chunks (batched to respect Cohere rate limits)."""
    embeddings = []
    batch_size = 96   # Cohere max per request
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        resp  = co.embed(
            texts=batch,
            model=EMBED_MODEL,
            input_type="search_document",
            embedding_types=["float"] # Explicitly request float embeddings
        )
        # Access the float_ attribute of the V2 embeddings object
        embeddings.extend(resp.embeddings.float_)
        
        print(f"  Embedded batch {i // batch_size + 1} "
              f"({min(i + batch_size, len(chunks))}/{len(chunks)} chunks)")
    return embeddings


def embed_query(question: str) -> list[float]:
    """Embed a single user query."""
    resp = co.embed(
        texts=[question],
        model=EMBED_MODEL,
        input_type="search_query",
        embedding_types=["float"]
    )
    # Extract the first embedding from the float_ array
    return resp.embeddings.float_[0]

## 5. Pinecone Vector Store
Creates a serverless index (free tier).

In [124]:
def get_or_create_index(index_name: str = INDEX_NAME,
                        dim: int = EMBED_DIM) -> object:
    """Create the Pinecone index if it doesn't exist, then return it."""
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        print(f"Creating Pinecone index '{index_name}' …")
        pc.create_index(
            name=index_name,
            dimension=dim,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        # Wait until ready
        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)
        print(f"Index '{index_name}' created.")
    else:
        print(f"Index '{index_name}' already exists.")
    return pc.Index(index_name)


def upsert_chunks(index, chunks: list[str], embeddings: list[list[float]]):
    """Upsert (id, vector, metadata) records into Pinecone."""
    records = [
        {
            "id": str(uuid.uuid4()),
            "values": emb,
            "metadata": {"text": chunk},
        }
        for chunk, emb in zip(chunks, embeddings)
    ]
    # Pinecone recommends batch upserts of ≤100
    batch_size = 100
    for i in range(0, len(records), batch_size):
        index.upsert(vectors=records[i : i + batch_size])
    print(f"Upserted {len(records)} vectors into Pinecone.")


def retrieve(index, query_embedding: list[float],
             k: int = TOP_K) -> list[tuple[str, float]]:
    """Query Pinecone and return top-k (chunk_text, score) pairs."""
    result = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True,
    )
    return [
        (match["metadata"]["text"], match["score"])
        for match in result["matches"]
    ]

## 6. Answer Generation with Cohere

The retrieved chunks form the grounded context. Cohere's command is prompted to answer strictly from that context.

In [125]:
def generate_answer(question: str,
                    retrieved: list[tuple[str, float]]) -> str:
    """Build a RAG prompt and call Cohere to generate the final answer."""
    context = "\n\n---\n\n".join(
        f"[Chunk {i+1} | score={score:.3f}]\n{chunk}"
        for i, (chunk, score) in enumerate(retrieved)
    )
    prompt = (
        "You are a precise document assistant. "
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say: "
        "'I could not find that information in the document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )
    
    # Use the V2 'messages' structure
    response = co.chat(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.2,
        stop_sequences=["--"],
    )
    
    # Iterate through the response content blocks to find the actual text
    for block in response.message.content:
        if block.type == "text":
            return block.text.strip()
            
    return "No valid text response generated."

## 7. Full RAG Pipeline — Orchestrator

In [126]:
class RAGPipeline:
    """
    End-to-end RAG system (Cohere + Pinecone).

    Usage
    -----
    rag = RAGPipeline()
    rag.ingest("my_doc.pdf")          # one-time indexing
    answer = rag.ask("What is X?")    # query any number of times
    """

    def __init__(self,
                 chunk_size: int = CHUNK_SIZE,
                 overlap: int = CHUNK_OVERLAP,
                 top_k: int = TOP_K):
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self.top_k      = top_k
        self.index      = None

    # Ingest 
    def ingest(self, path: str):
        """Load → chunk → embed (Cohere) → upsert (Pinecone)."""
        print("  INGESTION PIPELINE")
        print("="*50)

        # Stage 1: Load
        text = load_document(path)

        # Stage 2: Chunk
        chunks = chunk_text(text, self.chunk_size, self.overlap)

        # Stage 3: Embed
        print("\nEmbedding chunks with Cohere …")
        embeddings = embed_documents(chunks)

        # Stage 4: Index
        self.index = get_or_create_index()
        upsert_chunks(self.index, chunks, embeddings)

        print("\n Ingestion complete. Ready to answer questions!")

    # Query
    def ask(self, question: str, verbose: bool = True) -> str:
        """Embed query → retrieve (Pinecone) → generate (Cohere)."""
        if self.index is None:
            # Try to connect to an existing index
            self.index = pc.Index(INDEX_NAME)

        # Stage 5: Embed query
        q_vec = embed_query(question)

        # Stage 6: Retrieve
        results = retrieve(self.index, q_vec, k=self.top_k)

        if verbose:
            print(f"\n Retrieved {len(results)} chunks for: \"{question}\"")
            for i, (chunk, score) in enumerate(results, 1):
                preview = textwrap.shorten(chunk, width=110, placeholder=" …")
                print(f"  [{i}] score={score:.3f} | {preview}")

        # Stage 7: Generate
        answer = generate_answer(question, results)
        return answer

## 8. Create a Sample Document & Ingest

In [127]:
# Option A: Custom PDF
DOC_PATH = "Research_Paper_v3.pdf"   # ← change to your file path


# Option B: Quick demo with a text file
'''SAMPLE_TEXT = """
Retrieval-Augmented Generation (RAG) is a framework for improving large language model
outputs by incorporating external knowledge retrieval. Instead of relying solely on
parameters learned during pre-training, a RAG system first retrieves relevant documents
from an external corpus, then conditions the language model on the retrieved context
to produce the final answer.

The main advantage of RAG is factual grounding. Because the model is given actual source
text at inference time, it is far less likely to hallucinate. This is especially valuable
for enterprise and domain-specific applications where accuracy is critical.

RAG systems consist of three core components:
1. A retriever — typically a bi-encoder that embeds both queries and documents.
2. A vector store — a database for fast approximate nearest-neighbour search.
3. A generator — a language model that reads the retrieved context and produces an answer.

Popular embedding models used in RAG include OpenAI text-embedding-ada-002,
sentence-transformers/all-MiniLM-L6-v2, and Cohere embed-english-v3.0.
Common vector stores include FAISS (local), Pinecone (managed), Chroma, and Weaviate.

Chunking strategy is one of the most important design decisions in a RAG pipeline.
Chunks that are too large dilute relevance; chunks that are too small lose context.
A typical starting point is 200-400 words with a 10-20% overlap.

Hybrid search, which combines dense vector search with sparse BM25 keyword search,
often improves retrieval quality compared to using either method alone.
Re-ranking models such as cross-encoders can further boost precision by rescoring
the top-K retrieved candidates before they are passed to the generator.

Evaluation of RAG systems is typically done along three dimensions:
- Faithfulness: does the answer stay true to the retrieved context?
- Answer relevance: does the answer address the user's question?
- Context recall: were the most relevant chunks retrieved?
Tools such as RAGAs and TruLens automate this evaluation process.
"""


DOC_PATH = "sample_rag_document.txt"
with open(DOC_PATH, "w") as f:
    f.write(SAMPLE_TEXT)
print(f"Sample document saved to '{DOC_PATH}'")
'''

'SAMPLE_TEXT = """\nRetrieval-Augmented Generation (RAG) is a framework for improving large language model\noutputs by incorporating external knowledge retrieval. Instead of relying solely on\nparameters learned during pre-training, a RAG system first retrieves relevant documents\nfrom an external corpus, then conditions the language model on the retrieved context\nto produce the final answer.\n\nThe main advantage of RAG is factual grounding. Because the model is given actual source\ntext at inference time, it is far less likely to hallucinate. This is especially valuable\nfor enterprise and domain-specific applications where accuracy is critical.\n\nRAG systems consist of three core components:\n1. A retriever — typically a bi-encoder that embeds both queries and documents.\n2. A vector store — a database for fast approximate nearest-neighbour search.\n3. A generator — a language model that reads the retrieved context and produces an answer.\n\nPopular embedding models used in RAG in

In [128]:
# Build pipeline and ingest
rag = RAGPipeline(chunk_size=100, overlap=20, top_k=3)
rag.ingest(DOC_PATH)

  INGESTION PIPELINE
Loaded 'Research_Paper_v3.pdf' — 24,832 characters
Created 43 chunks (size=100 words, overlap=20 words)

Embedding chunks with Cohere …
  Embedded batch 1 (43/43 chunks)
Index 'rag-week7' already exists.
Upserted 43 vectors into Pinecone.

 Ingestion complete. Ready to answer questions!


## 9. Q&A

In [135]:
questions = [
    "What are the three specific cryptographic primitives combined in the hybrid KEM-DEM biometric data vault design?",
    "Why does the ML-DSA-44 signature size cause issues with IP packet fragmentation, and how does this impact TCP Retransmission Timeouts (RTO) over LEO satellite links?",
    "What is the architectural purpose of separating the computationally heavy post-quantum key exchange from the bulk symmetric AES-256-GCM encryption in this vault design?"
]

for q in questions:
    print(f" \n Question:\n{q}")
    print("-"*80)
    answer = rag.ask(q, verbose=True)
    print(f"\n Answer:\n{textwrap.fill(answer, width=80)}")

 
 Question:
What are the three specific cryptographic primitives combined in the hybrid KEM-DEM biometric data vault design?
--------------------------------------------------------------------------------

 Retrieved 3 chunks for: "What are the three specific cryptographic primitives combined in the hybrid KEM-DEM biometric data vault design?"
  [1] score=0.771 | is more than 20 MB [1]. This work contributes to the existing research through four main contributions: (i) …
  [2] score=0.748 | of the server private key of the ML-KEM scheme. V. CONCLUSION It can be seen that the architecture of the …
  [3] score=0.694 | with ridge angle and type make up a multidimensional vector that is analysed using CNNs for ridge …

 Answer:
The three specific cryptographic primitives combined in the hybrid KEM-DEM
biometric data vault design are FIPS 203, FIPS 204, and AES-256-GCM.
 
 Question:
Why does the ML-DSA-44 signature size cause issues with IP packet fragmentation, and how does this impact T

## 10. Single Custom Query


In [130]:
my_question = "How many biometric storage operations per second can an 8-core server process using this architecture?"
answer = rag.ask(my_question)
print(f"Q: {my_question}\nA: {answer}")


 Retrieved 3 chunks for: "How many biometric storage operations per second can an 8-core server process using this architecture?"
  [1] score=0.726 | IV. DISCUSSION A. Computational Performance Readiness The benchmark results confirm that the hardware- …
  [2] score=0.639 | IV. DISCUSSION A. Computational Performance Readiness The benchmark results confirm that the hardware- …
  [3] score=0.626 | is more than 20 MB [1]. This work contributes to the existing research through four main contributions: (i) …
Q: How many biometric storage operations per second can an 8-core server process using this architecture?
A: An 8-core server can process over 50,000 biometric store operations per second using this architecture.


## 11. Experiment - Chunk Size Ablation

Compare how different chunk sizes affect the retrieved context and final answer quality.

In [136]:
TEST_Q = "Why does the ML-DSA-44 signature size cause issues with IP packet fragmentation, and how does this impact TCP Retransmission Timeouts (RTO) over LEO satellite links?"

for chunk_size, overlap in [(50, 10), (100, 20), (200, 40)]:
    print(f"\n{'='*60}")
    print(f"chunk_size={chunk_size}, overlap={overlap}")
    print("="*60)

    # Delete and recreate index for a clean experiment
    if INDEX_NAME in [i.name for i in pc.list_indexes()]:
        pc.delete_index(INDEX_NAME)
        time.sleep(2)

    exp = RAGPipeline(chunk_size=chunk_size, overlap=overlap, top_k=3)
    exp.ingest(DOC_PATH)
    ans = exp.ask(TEST_Q, verbose=False)
    print(f"\n {textwrap.fill(ans, 72)}")


chunk_size=50, overlap=10
  INGESTION PIPELINE
Loaded 'Research_Paper_v3.pdf' — 24,832 characters
Created 85 chunks (size=50 words, overlap=10 words)

Embedding chunks with Cohere …
  Embedded batch 1 (85/85 chunks)
Creating Pinecone index 'rag-week7' …
Index 'rag-week7' created.
Upserted 85 vectors into Pinecone.

 Ingestion complete. Ready to answer questions!

 The ML‑DSA‑44 signature is 2,420 bytes long, which is larger than the
standard Ethernet MTU of 1,500 bytes. Because the signature cannot fit
in a single MTU packet, it must be split into multiple IP fragments. In
a high‑loss, high‑latency LEO satellite environment, these fragmented
packets experience a higher probability of loss and additional delay.
The regression analysis shows that this fragmentation leads to a 51 %
probability of catastrophic TCP Retransmission Timeouts (RTOs) that
exceed 1,000 ms per authentication handshake, significantly increasing
the likelihood of connection failures over LEO satellite links.

chunk

## 12. Clean Up (Optional)

Delete the Pinecone index when done to stay within the free-tier limit.

In [132]:
# Uncomment to delete the index
#pc.delete_index(INDEX_NAME)
#print(f"Index '{INDEX_NAME}' deleted.")